# Water Leak Detection — LSTM Autoencoder Demo

Interactive walkthrough of the full anomaly-detection pipeline.

**Sensors**: `pressure_bar`, `flow_rate_lps`, `temperature_c`, `vibration_ms2`  
**Leak events**: injected at timesteps 1500, 3500, 5800  
**Architecture**: LSTM Autoencoder — reconstruction error threshold  
**Alert engine**: NORMAL → SUSPICIOUS → ALERT → CONFIRMED state machine

In [ ]:
import sys, pathlib
# Add project root to path when running from notebooks/
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from data.generate_sensor_data import generate_sensor_data, SENSOR_COLS, LEAK_EVENTS
from pipeline.preprocessing import SensorPreprocessor
from pipeline.detector import AnomalyDetector
from pipeline.alert_engine import AlertEngine, AlertState

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print('Imports OK')

## 1. Generate Synthetic Sensor Data

In [ ]:
df = generate_sensor_data(n_timesteps=6000, seed=SEED)
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
for ax, col in zip(axes, SENSOR_COLS):
    ax.plot(df['timestep'], df[col], lw=0.6)
    ax.set_ylabel(col, fontsize=8)
    for start, dur in LEAK_EVENTS:
        ax.axvspan(start, start + dur, alpha=0.2, color='red', label='leak')
axes[-1].set_xlabel('Timestep')
fig.suptitle('Raw Sensor Readings (red = injected leak)', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Pre-process — Sliding Windows (70 / 15 / 15 split)

In [ ]:
prep = SensorPreprocessor(seq_len=50, train_frac=0.70, val_frac=0.15, step=1)
X_train, X_val, X_test, y_train, y_val, y_test = prep.fit_transform(df)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Anomaly rate  — train:{y_train.mean():.3f}  val:{y_val.mean():.3f}  test:{y_test.mean():.3f}')

## 3. Train LSTM Autoencoder

In [ ]:
detector = AnomalyDetector(
    n_features=4, hidden_dim=64, latent_dim=16, n_layers=2,
    seq_len=50, dropout=0.2, lr=1e-3, batch_size=256,
    n_epochs=60, patience=8, threshold_k=3.0
)
train_losses = detector.fit(X_train, y_train, X_val, y_val)

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(train_losses, lw=1.2)
plt.xlabel('Epoch')
plt.ylabel('Train MSE')
plt.title('Training Loss Curve')
plt.tight_layout()
plt.show()

## 4. Evaluate on Test Set

In [ ]:
metrics = detector.evaluate(X_test, y_test, split_name='test')
print(metrics)

In [ ]:
test_errors, test_anomalies = detector.predict(X_test)
SEQ_LEN, STEP = 50, 1
test_ts_start = (len(X_train) + len(X_val)) * STEP + SEQ_LEN - 1
test_ts = np.arange(len(test_errors)) * STEP + test_ts_start

plt.figure(figsize=(14, 3))
plt.plot(test_ts, test_errors, lw=0.7, color='steelblue', label='Recon error')
plt.axhline(detector.threshold, color='red', ls='--', lw=1.5, label=f'Threshold={detector.threshold:.4f}')
for start, dur in LEAK_EVENTS:
    plt.axvspan(start, start + dur, alpha=0.18, color='crimson')
plt.xlabel('Timestep')
plt.ylabel('MSE')
plt.title('Reconstruction Error on Test Set')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Alert Engine — State Machine

In [ ]:
engine = AlertEngine(suspicious_window=3, alert_window=6, recovery_window=5)
state_arr = engine.run(test_errors, test_anomalies,
                       timestep_offset=len(X_train) + len(X_val))
engine.summary()

In [ ]:
STATE_COLORS = ['#4caf50', '#ff9800', '#f44336', '#9c27b0']
STATE_LABELS = ['NORMAL', 'SUSPICIOUS', 'ALERT', 'CONFIRMED']
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(test_ts, test_errors, lw=0.7, color='#ff7f0e')
axes[0].axhline(detector.threshold, color='red', ls='--', lw=1.4)
axes[0].set_ylabel('Recon Error')
axes[0].set_title('Alert Engine State Timeline')

ax = axes[1]
for i in range(len(state_arr) - 1):
    ax.fill_between([test_ts[i], test_ts[i+1]], [0,0], [state_arr[i]+1, state_arr[i]+1],
                    color=STATE_COLORS[state_arr[i]], step='pre', linewidth=0)
ax.set_yticks([1,2,3,4])
ax.set_yticklabels(STATE_LABELS, fontsize=8)
ax.set_xlabel('Timestep')
ax.set_ylabel('State')
patches = [mpatches.Patch(color=STATE_COLORS[i], label=STATE_LABELS[i]) for i in range(4)]
ax.legend(handles=patches, ncol=4, fontsize=8)
plt.tight_layout()
plt.show()